In [53]:
# dotenv initialization
import dotenv
import os

dotenv.load_dotenv()

True

In [54]:
import pandas as pd
from langchain_core.documents import Document

data = pd.read_csv("dataset/Tamil_movies_dataset.csv")
df = pd.read_csv("dataset/tamil.csv") 

def generate_unified_profile(row):
    name = row.get('MovieName') or row.get('Title')
    genre = row.get('Genre')
    director = row.get('Director')
    actor = row.get('Actor') or row.get('Cast')
    year = row.get('Year') or row.get('Release Year')
    rating = row.get('Rating')
    plot = row.get('Plot', 'No plot available')

    return (
        f"Title: {name}\n"
        f"Genre: {genre}\n"
        f"Director: {director}\n"
        f"Actor: {actor}\n"
        f"Release Year: {year}\n"
        f"Rating: {rating}\n"
        f"Synopsis: {plot}\n"
    )

data["profile"] = data.apply(generate_unified_profile, axis=1)
docs_1 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_movies_dataset",
            "title": row.get('MovieName'),
            "genre": row.get('Genre'),
            "year": row.get('Year')
        }
    ) for _, row in data.iterrows()
]

df["profile"] = df.apply(generate_unified_profile, axis=1)
docs_2 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_dataset_2",
            "title": row.get('Title'),
            "genre": row.get('Genre'),
            "year": row.get('Release Year')
        }
    ) for _, row in df.iterrows()
]

documents = docs_1 + docs_2

print(f"Total documents prepared for TrailerCraft: {len(documents)}")


Total documents prepared for TrailerCraft: 745


In [55]:
# Embeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(documents,embeddings,persist_directory="dataset/Tamil_movies_dataset_chroma")

In [56]:
# Output Classes
from pydantic import BaseModel,Field
from typing import List

class DirectorConsultation(BaseModel):
    style_tip: str = Field(description="A tip on how to adapt the director's signature filmmaking style.")
    trademark_interval: str = Field(description="A description of a signature interval block typical for this director's style.")

class Shot(BaseModel):
    shotnumber:int
    visual:str=Field(description="A brief description of the visual content of the shot.")
    cameraangle:str=Field(description="The camera angle used in the shot, e.g., close-up, wide shot, aerial view.")
    audio_cue:str=Field(description="Any significant audio cues present in the shot, such as dialogue, sound effects, or music.")

class TrailerPackage(BaseModel):
    structure: str = Field(description="The 3-act breakdown of the trailer")
    voice_over: str = Field(description="The script for the narrator it should be in tamil and that tamil is not pure it should be a tamil in a way that it is used in common")
    music_mood: str = Field(description="Instrumentation, tempo, and vibe")
    fonrstyle: str = Field(description="The font style to be used in the trailer")
    title: str = Field(description="The title of the movie in tamil and english that is good make it catchy and appealing")
    shot_list: List[Shot]
    director_consultation: DirectorConsultation


In [57]:
# Prompt template for the model
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

retrivar = vectorstore.as_retriever(search_kwargs={"k": 10})

prompt =ChatPromptTemplate.from_messages([
    ("system",    """You are a Kollywood Trailer Editor who is an expert in creating engaging and captivating trailers for Tamil movies.
    Your task is to analyze the provided movie plot and generate a detailed trailer structure that includes a 3-act breakdown, voice-over script, music mood, and a shot list with descriptions of visuals, camera angles, and audio cues using the synopsis of the movie also use the conservation hsitory as well"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", """
    CONTEEXT FROM DATABASE: {context}
    MOVIE SYNOPSIS: {synopsis}
    ADDITIONAL STYLE GUIDELINES: {instructions}
    """)
])

In [58]:
# LLM Integration with Groq and Gemini
import json
import re
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_classic.memory import ChatMessageHistory
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),temperature=0.7)
chat_history=ChatMessageHistory()

def segregate_intent(user_input, history):
    parser_prompt = f"""
    Analyze this user request: "{user_input}"
    
    Rules:
    - If the user asks for a specific tiny detail (just a title, just BGM, just a cast idea) without wanting a full script, set mode: "specific".
    - If the user asks for a trailer, a script, or a full story breakdown, set mode: "full".
    
    Return ONLY JSON: 
    {{
        "synopsis": "extracted plot", 
        "mode": "full" or "specific", 
        "target": "title" or "bgm" or "script",
        "style": "extracted style",
        "instructions": "any other notes"
    }}
    """
    raw_response = model.invoke(parser_prompt).content
    clean_json = re.sub(r"```json|```", "", raw_response).strip()
    return json.loads(clean_json)

def generate_trailer_package(user_input: str) -> TrailerPackage:
    parsed = segregate_intent(user_input, chat_history.messages)
    search_query = parsed.get('synopsis') or user_input
    relevant_docs = retrivar.invoke(search_query)
    context_data = "\n\n".join([doc.page_content for doc in relevant_docs])
    if parsed['mode'] == 'specific':
        specific_prompt = f"Context: {context_data}\nHistory: {chat_history.messages}\nUser: {user_input}\nAnswer ONLY the specific request briefly."
        response = model.invoke(specific_prompt).content
        print(response)
        chat_history.add_user_message(user_input)
        chat_history.add_ai_message(response)
        return response
    else:
        structeredllm = model.with_structured_output(TrailerPackage)
        chain = prompt | structeredllm
        response = chain.invoke({
            "chat_history": chat_history.messages,
            "context": context_data,
            "synopsis": parsed['synopsis'],
            "instructions": f"Style: {parsed.get('style', 'General')}. Extra: {parsed.get('instructions', 'None')}"
        })
        
        _print_formatted_output(response)
        
        chat_history.add_user_message(user_input)
        chat_history.add_ai_message(f"Generated trailer: {response.title}")
                
        return response

def _print_formatted_output(response):
    data = response.model_dump()
    for key, value in data.items():
        if key == "director_consultation": continue
        print(f"\n{key.replace('_', ' ').upper()}")
        if isinstance(value, list):
            i=1
            for item in value:
                print(f"Shot {item.get('shot_number', i)}:\n{item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}")
                i+=1
        else:
            print(value)
    
    print("\nDIRECTOR CONSULTATION")
    print(f"Tip: {response.director_consultation.style_tip}")
    print(f"Interval: {response.director_consultation.trademark_interval}")

In [59]:
response=generate_trailer_package("I want a trailer for a Tamil romantic comedy with a twist of mystery, set in Chennai. The trailer should have a vibrant and energetic vibe, with a mix of humor and suspense. The title should be catchy and appealing in both Tamil and English.")


STRUCTURE
Act 1: The Dream & The Disaster - Introduce Harish, his filmmaking ambition, and the journey to Singapore. Establish the initial comedic tone with his misfortunes, culminating in the loss of his passport.Act 2: The Unexpected Alliance & New Horizons - Introduce Vaanambaadi, the eccentric cameraman, and their quirky partnership. Show Harish's renewed hope for his film. Transition to the introduction of the girl with cancer, hinting at the shift to more serious, yet hilarious, situations.Act 3: The Crossroads & The Message - Build emotional stakes around the girl's condition and Harish's involvement. Showcase montage of humorous and poignant moments exploring themes of life, death, and human connection. Conclude with an uplifting, impactful sequence that leaves the audience pondering the journey's profound impact.

VOICE OVER
VO: 'ஒரு சினிமா கனவோட, ஒரு டிக்கெட்டோட, கிளம்பினான் ஹரிஷ்... சென்னைல இருந்து சிங்கப்பூருக்கு!' (Oru cinema kanavoda, oru ticketoda, kilambinaan Harish...

In [60]:
repsonse = generate_trailer_package("Just give me a catchy title in Tamil and English for a romantic comedy with a twist of mystery, set in Chennai.")

Tamil: சென்னை காதல் மர்மம்
English: Chennai Love Mystery


In [61]:
response3=generate_trailer_package("Can you give me a style tip on how to adapt the director's signature filmmaking style for this romantic comedy with a twist of mystery, set in Chennai?")

Emphasize humor arising from the protagonist's misfortunes and eccentric encounters as they navigate both an unconventional romance and the unfolding mystery, maintaining a lighthearted, madcap tone even amidst serious plot points.


In [62]:
response4 = generate_trailer_package("just give me the style of bgms for a gangster movie")

Gritty, intense, and atmospheric, often incorporating dark orchestral elements, heavy basslines, unsettling synths, and percussive rhythms to build tension and emphasize power struggles.


In [63]:
respine=generate_trailer_package("just give me the style of bgms")

Dynamic, high-energy, and suspenseful, often featuring powerful orchestral scores, pulsating electronic beats, driving percussion, and intense sound design.
